# Допуск: Efficient Estimation of Word Representations in Vector Space

**Статья:** T. Mikolov, K. Chen, G. Corrado, J. Dean, 2013 — <https://arxiv.org/abs/1301.3781>

5 заданий. Под заголовком каждого задания указан его тип:

- **что есть что** — названия функций или классов заменены: определите, что чему
  соответствует, и ответьте на вопросы;
- **расхождение со статьёй** — код реализует не то, что описано в статье: найдите это место;
- **перепутаны местами блоки кода** — какой-то блок стоит не на своём месте: найдите его;
- **неверный комментарий** — код корректен, но один комментарий утверждает неправду: найдите
  его.

Для найденного отклонения напишите: где оно, что не так, какое место статьи (раздел /
формула / таблица / цитата) нарушено и как должно быть. Код писать не нужно — достаточно
объяснить словами. Запускать и отлаживать код тоже не нужно: все задания решаются чтением
кода и статьи.

Общие импорты:

In [ ]:
import math, random
import torch
import torch.nn as nn
import torch.nn.functional as F

`ids` — одномерный `LongTensor` с индексами слов корпуса.

## Задание 1. Нарезка обучающих примеров

**Тип задания:** что есть что — названия функций/классов заменены, определите, что чему соответствует.

Четыре функции готовят обучающие примеры для Skip-gram, CBOW, feedforward NNLM и RNNLM.
Определите, какая функция для какой модели.

In [ ]:
def make_a(ids, k=4):
    X = torch.stack([ids[i - k:i] for i in range(k, len(ids))])
    return X, ids[k:]                                # (B, k), (B,)

def make_b(ids, k=8):
    n = (len(ids) - 1) // k
    X = ids[:n * k].view(n, k)
    Y = ids[1:n * k + 1].view(n, k)
    return X, Y                                      # (B, k), (B, k)

def make_c(ids, k=5):
    X, Y = [], []
    for i in range(len(ids)):
        R = random.randint(1, k)
        for j in range(max(0, i - R), min(len(ids), i + R + 1)):
            if j != i:
                X.append(ids[i]); Y.append(ids[j])
    return torch.stack(X), torch.stack(Y)            # (B,), (B,)

def make_d(ids, k=2):
    X, Y = [], []
    for i in range(k, len(ids) - k):
        X.append(torch.cat([ids[i - k:i], ids[i + 1:i + k + 1]]))
        Y.append(ids[i])
    return torch.stack(X), torch.stack(Y)            # (B, 2k), (B,)

**Вопрос.** Зачем в одной из функций случайное `R`? Как это связано с фразой из §3.2
«we give less weight to the distant words»?

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 2. Четыре архитектуры

**Тип задания:** что есть что — названия функций/классов заменены, определите, что чему соответствует.

Четыре класса реализуют Skip-gram, CBOW, NNLM и RNNLM (для краткости выход везде — полный
softmax). Определите, какой класс какая модель.

In [ ]:
class ModelA(nn.Module):
    def __init__(self, V, D):
        super().__init__()
        self.emb, self.out = nn.Embedding(V, D), nn.Linear(D, V, bias=False)
    def forward(self, x):                  # (B, 2W)
        return self.out(self.emb(x).mean(1))                  # (B, V)

class ModelB(nn.Module):
    def __init__(self, V, H):
        super().__init__()
        self.emb, self.out = nn.Embedding(V, H), nn.Linear(H, V)
        self.rnn = nn.RNN(H, H, batch_first=True)
    def forward(self, x):                  # (B, T)
        return self.out(self.rnn(self.emb(x))[0])             # (B, T, V)

class ModelC(nn.Module):
    def __init__(self, V, D, N, H):
        super().__init__()
        self.emb = nn.Embedding(V, D)
        self.hidden, self.out = nn.Linear(N * D, H), nn.Linear(H, V)
    def forward(self, x):                  # (B, N)
        p = self.emb(x).flatten(1)                            # (B, N*D)
        return self.out(torch.tanh(self.hidden(p)))           # (B, V)

class ModelD(nn.Module):
    def __init__(self, V, D):
        super().__init__()
        self.emb, self.out = nn.Embedding(V, D), nn.Linear(D, V, bias=False)
    def forward(self, x):                  # (B,)
        return self.out(self.emb(x))                          # (B, V)

**Вопрос.** Чем две модели без скрытого слоя (log-linear, §3) отличаются от двух других и
что авторы этим выиграли?

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 3. Стоимость обучения

**Тип задания:** неверный комментарий — код корректен, но один комментарий утверждает неправду.

Функция считает стоимость обучения по формулам (1)–(5) статьи. Формулы записаны верно,
но **один из комментариев** неверен по смыслу — найдите его.

In [ ]:
def train_cost(model, E, T, V, D, N=10, H=500, C=10):
    # O = E * T * Q, Q = parameters touched per training example (Sec. 2)
    Q = {
        "nnlm":     N * D + N * D * H + H * math.log2(V),   # hidden layer dominates
        "rnnlm":    H * H + H * math.log2(V),               # recurrent matrix dominates
        "cbow":     N * D + D * math.log2(V),               # no hidden layer at all
        "skipgram": C * (D + D * math.log2(V)),             # cheaper than CBOW: one input word
    }[model]
    return E * T * Q

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 4. Коды слов для иерархического softmax

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Функция строит двоичные коды слов (пути в дереве) для иерархического softmax. В коде
**одно отклонение** от статьи — найдите его.

In [ ]:
def word_codes(counts):
    # counts[w] = corpus frequency of word w
    # returns a binary code (root -> leaf branch choices) for every word
    V = len(counts)
    L = math.ceil(math.log2(V))
    return [[(w >> k) & 1 for k in reversed(range(L))] for w in range(V)]

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 5. Цикл обучения

**Тип задания:** перепутаны местами блоки кода.

`encoder` возвращает вектор `h` (для Skip-gram это просто `nn.Embedding(V, D)`), `hs` —
выходной слой. В коде **одно отклонение** — найдите его.

In [ ]:
def train(encoder, hs, X, Y, epochs=1, lr0=0.025, B=256):
    opt = torch.optim.SGD([*encoder.parameters(), *hs.parameters()], lr=lr0)
    total, step = epochs * math.ceil(len(X) / B), 0
    for epoch in range(epochs):
        perm = torch.randperm(len(X))
        for i in range(0, len(X), B):
            idx = perm[i:i + B]
            for g in opt.param_groups:                    # linear decay to zero
                g["lr"] = lr0 * (1 - step / total)
            loss = hs(encoder(X[idx]), Y[idx]).sum()
            loss.backward()
            opt.zero_grad()
            opt.step()
            step += 1

**Ваш ответ:**

_Напишите ответ здесь._